# Week 4: HealthConnect Clinic — Machine Learning Problem Definition

## Project: HealthConnect Experience Lab
## Track: Data Science
## Central Business Question: How can HealthConnect Clinic use data and AI to reduce missed appointments and improve the patient support experience?

This notebook defines the machine learning problem for predicting 
patient appointment no-shows, based on the HealthConnect Appointment 
Dataset. This is a foundation-stage deliverable; no model is built or 
deployed at this stage.

## Task 1: Dataset Review

The HealthConnect Appointment Dataset is loaded and reviewed to 
understand its structure, size, and the variables available for 
analysis.

In [10]:
import pandas as pd

df = pd.read_csv(r"C:\Users\HP\Downloads\HealthConnect_Appointment_Data.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (5000, 18)

Columns:
['appointment_id', 'patient_id', 'gender', 'age', 'age_group', 'appointment_type', 'booking_date', 'appointment_date', 'appointment_day', 'appointment_time', 'booking_lead_days', 'previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'waiting_time_minutes', 'appointment_outcome']


In [11]:
df.head(10)

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show
5,HC-00006,P-0523,Male,66,65+,Specialist Consultation,2/16/2026,4/14/2026,Tuesday,Morning,57,4,2,No,NaN,3.1,36.0,No-Show
6,HC-00007,P-0776,Male,58,55-64,Diagnostic Test,1/17/2025,2/16/2025,Sunday,Afternoon,30,4,1,Yes,WhatsApp,8.2,30.0,No-Show
7,HC-00008,P-0927,Female,28,25-34,Specialist Consultation,11/27/2025,1/16/2026,Friday,Afternoon,50,6,1,Yes,SMS,19.2,21.0,Attended
8,HC-00009,P-0388,Male,77,65+,Follow-up,3/9/2025,4/21/2025,Monday,Afternoon,43,3,0,Yes,SMS,10.1,27.0,No-Show
9,HC-00010,P-0371,Female,20,18-24,Follow-up,2/5/2025,2/21/2025,Friday,Afternoon,16,2,0,Yes,SMS,28.0,20.0,Attended


### Dataset Overview

The dataset contains **5,000 appointment records** across **18 
variables**, with one row per appointment. It includes patient 
demographics, appointment characteristics, booking information, 
patient history, reminder details, and the final appointment outcome.

The dataset also contains **1,696 unique patients**, meaning many 
patients appear across multiple appointments. This is relevant for 
later modelling decisions (see Key Modelling Considerations).

## Task 2: Data Quality Assessment

The dataset is assessed for missing values, duplicate records, 
logical consistency, and the distribution of the target variable 
before any modelling decisions are made.

In [12]:
print("=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Records ===")
print("Full duplicate rows:", df.duplicated().sum())
print("Duplicate appointment_id:", df['appointment_id'].duplicated().sum())

print("\n=== Target Variable Distribution: appointment_outcome ===")
print(df['appointment_outcome'].value_counts())
print(df['appointment_outcome'].value_counts(normalize=True) * 100)

=== Missing Values ===
appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

=== Duplicate Records ===
Full duplicate rows: 0
Duplicate appointment_id: 0

=== Target Variable Distribution: appointment_outcome ===
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64
appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64


In [13]:
print("=== Logical Consistency Checks ===")
print("previous_no_shows exceeding previous_appointments:",
      (df['previous_no_shows'] > df['previous_appointments']).sum())
print("Negative booking_lead_days:", (df['booking_lead_days'] < 0).sum())

print("\n=== reminder_channel when reminder_sent = No ===")
print(df[df['reminder_sent'] == 'No']['reminder_channel'].value_counts(dropna=False))

print("\nUnique patients:", df['patient_id'].nunique())
print("Repeat patient rows:", df['patient_id'].duplicated().sum())

=== Logical Consistency Checks ===
previous_no_shows exceeding previous_appointments: 0
Negative booking_lead_days: 0

=== reminder_channel when reminder_sent = No ===
reminder_channel
NaN    1366
Name: count, dtype: int64

Unique patients: 1696
Repeat patient rows: 3304


### Data Quality Findings

| Issue | Detail |
|---|---|
| Missing `reminder_channel` | 1,366 rows (27%) - but every instance corresponds exactly to `reminder_sent = No`, so this is structurally expected rather than a data quality problem |
| Missing `distance_to_clinic_km` | 90 rows (1.8%) - genuine missingness |
| Missing `waiting_time_minutes` | 60 rows (1.2%) - genuine missingness |
| Duplicate records | None found (0 full duplicates, 0 duplicate appointment IDs) |
| Logical consistency | `previous_no_shows` never exceeds `previous_appointments`; no negative `booking_lead_days` - both pass sanity checks |
| Target class balance | No-Show 48.5%, Attended 46.3%, Cancelled 5.3% |
| Repeat patients | 3,304 of 5,000 rows belong to a patient with more than one appointment in the dataset |

Overall, the dataset is clean and well-structured. The main data 
quality decision required is how to treat the `Cancelled` category 
within `appointment_outcome`, addressed in the Target Variable section 
below.

## Task 3: Machine Learning Problem Definition

### Business Context
HealthConnect Clinic experiences a high rate of missed appointments 
(no-shows), which leads to wasted appointment slots, reduced clinic 
efficiency, and disrupted care for patients who might otherwise have 
used that slot. The clinic wants to use data and machine learning to 
better understand and anticipate which appointments are at risk of 
being missed.

### Problem Type
This is a **binary classification problem**. Given information known 
at the time an appointment is booked or shortly before it occurs, the 
goal is to predict whether a patient will attend or not attend 
(no-show) their scheduled appointment.

### Machine Learning Objective
Predict the likelihood that a given appointment will result in a 
no-show, so that HealthConnect Clinic can proactively intervene (for 
example, through targeted reminders, rebooking outreach, or 
overbooking strategies) for appointments flagged as high-risk.

### Scope Boundary
This problem is scoped specifically to **no-shows**, not 
cancellations. A cancellation represents a patient proactively 
communicating their inability to attend, which allows the clinic an 
opportunity to rebook the slot. A no-show is an unannounced absence, 
which is the actual operational cost the clinic is trying to reduce. 
Cancelled appointments are therefore excluded from the modelling 
dataset (see Target Variable section), and would be better addressed 
as a separate problem if explored in a later phase.

### Why This Is a Suitable Machine Learning Problem
- The outcome (`appointment_outcome`) is a recorded, historical label, 
  making this a supervised learning problem.
- The dataset contains variables plausibly related to attendance 
  behavior (booking lead time, prior no-show history, reminders, 
  distance to clinic), giving the model meaningful signal to learn 
  from.
- The business use case (flagging at-risk appointments) is 
  actionable: a prediction can directly inform a clinic intervention 
  before the appointment date, which is what makes this genuinely 
  useful rather than just descriptive.

## Task 4: Proposed Target Variable

**Variable:** `appointment_outcome`, transformed into a binary label:
- 1 = No-Show
- 0 = Attended

### Task 6: Handling of Cancelled Appointments

Rows where `appointment_outcome = Cancelled` (263 rows, 5.3% of the 
dataset) are excluded from the modelling dataset. Cancellations 
represent a distinct patient behavior (proactive communication) from 
a no-show (silent absence), and including them would blur the target 
rather than strengthen it. This leaves **4,737 rows** for modelling, 
with a well-balanced target distribution (~51% No-Show, ~49% 
Attended).

In [14]:
# Build the modelling dataset: exclude Cancelled, encode target as binary
df_model = df[df['appointment_outcome'] != 'Cancelled'].copy()
df_model['target_no_show'] = (df_model['appointment_outcome'] == 'No-Show').astype(int)

print("Modelling dataset shape:", df_model.shape)
print("\nTarget distribution:")
print(df_model['target_no_show'].value_counts())
print(df_model['target_no_show'].value_counts(normalize=True) * 100)

Modelling dataset shape: (4737, 19)

Target distribution:
target_no_show
1    2423
0    2314
Name: count, dtype: int64
target_no_show
1    51.150517
0    48.849483
Name: proportion, dtype: float64


## Task 5: Potential Input Features

Based on the dataset review, the following variables are considered 
potential input features for predicting no-shows, grouped by what 
they represent.

**Patient Demographics**
- `gender`
- `age` (or `age_group`)

**Appointment Characteristics**
- `appointment_type`
- `appointment_day`
- `appointment_time`
- `booking_lead_days` (days between booking and the appointment date)

**Patient History**
- `previous_appointments`
- `previous_no_shows`
- A derived feature: prior no-show rate 
  (`previous_no_shows / previous_appointments`), which may be more 
  informative than either raw count alone

**Reminder Information**
- `reminder_sent`
- `reminder_channel`

**Logistics**
- `distance_to_clinic_km`

### Features Excluded from Modelling

| Variable | Reason for Exclusion |
|---|---|
| `appointment_id` | Unique identifier, carries no predictive information |
| `patient_id` | Identifier, not a feature (though useful for grouping/validation, addressed below) |
| `booking_date` / `appointment_date` | Raw dates are not directly usable; `booking_lead_days` and `appointment_day` already extract the useful signal from them |
| `waiting_time_minutes` | Flagged as a likely data leakage risk (see Key Modelling Considerations) |

## Task 7: Initial Modelling Approach

**Step 1: Data Preparation**
- Filter out Cancelled appointments to build the modelling dataset 
  (4,737 rows). Completed above.
- Handle missing values in `distance_to_clinic_km` (90 rows) and 
  `waiting_time_minutes` (60 rows), if the latter is retained.
- Encode categorical variables (`gender`, `appointment_type`, 
  `appointment_day`, `appointment_time`, `reminder_sent`, 
  `reminder_channel`).
- Engineer the prior no-show rate feature described above.

**Step 2: Train/Test Split**
- Split the data into training and test sets (e.g. 80/20), using 
  stratification on the target variable to preserve the 
  No-Show/Attended balance in both sets.
- Because the dataset contains repeat patients (1,696 unique patients 
  across 5,000 appointments), a patient-level split will be 
  considered to avoid the same patient's appointments appearing in 
  both the training and test sets, which could otherwise inflate 
  performance estimates.

**Step 3: Baseline Modelling**
- Start with an interpretable baseline model (e.g. Logistic 
  Regression) to establish a benchmark and understand which features 
  carry signal.
- Progress to a tree-based model (e.g. Random Forest or Gradient 
  Boosting) if the baseline suggests non-linear relationships are 
  present.

**Step 4: Evaluation**
- Given the roughly balanced target, accuracy is a reasonable 
  starting metric, but precision, recall, and F1-score (particularly 
  for the No-Show class) will be prioritized, since the clinic's real 
  interest is correctly identifying at-risk appointments rather than 
  overall accuracy alone.

### Key Modelling Considerations

- **Potential data leakage:** `waiting_time_minutes` is very likely 
  recorded during or after the appointment (how long the patient 
  waited once they arrived), which means it would not be known at 
  prediction time for a future appointment. Including it would leak 
  information that implicitly confirms attendance. This will be 
  excluded from modelling unless it can be confirmed to be knowable 
  in advance.
- **Repeat patients:** Since many patients appear multiple times, care 
  is needed to prevent the same patient's data from leaking between 
  training and test sets.
- **Class imbalance in Cancelled:** Since Cancelled rows are excluded 
  rather than modelled, the clinic will not have a model for 
  predicting cancellations specifically; this is noted as a 
  limitation, not an oversight.
- **Missing values:** `distance_to_clinic_km` and 
  `waiting_time_minutes` have limited missingness (under 2%), which is 
  manageable through standard imputation if needed, and unlikely to 
  materially affect results.

## Assumptions, Limitations, Risks and Dependencies

### Assumptions
- It is assumed that the historical patterns in this dataset (e.g. the 
  relationship between reminders, booking lead time, and attendance) 
  reflect real-world patient behavior closely enough to be useful, 
  given that the data is synthetic.
- It is assumed that `previous_appointments` and `previous_no_shows` 
  are calculated consistently for each patient up to the point of the 
  current appointment, rather than including future appointments.
- It is assumed that `waiting_time_minutes` is not available at 
  prediction time and should therefore be treated as a leakage risk 
  rather than a usable feature, pending confirmation from the Data 
  Dictionary or project team.

### Limitations
- The dataset is synthetic and anonymised. Patterns learned from it 
  may not fully generalise to real HealthConnect patient behavior if 
  this project were ever applied to genuine clinic data.
- The dataset provides a single snapshot of appointment records rather 
  than a continuous stream, so seasonal or long-term trends in 
  attendance cannot be assessed.
- No socioeconomic, employment, or health-condition information is 
  available, all of which could plausibly influence attendance in a 
  real clinical setting but are absent here.
- Cancelled appointments are excluded from the modelling target, so 
  this solution does not address cancellation behavior, only no-shows.

### Risks
- **Data leakage risk:** as noted above, including 
  `waiting_time_minutes` could artificially inflate model performance 
  in a way that would not hold up in real-world deployment.
- **Patient-level leakage risk:** without a patient-aware train/test 
  split, the model could appear more accurate than it truly is by 
  learning individual patient identity rather than generalisable 
  attendance patterns.
- **Bias risk:** if certain demographic groups are historically 
  flagged as high-risk for no-shows, a deployed model could reinforce 
  differential treatment (e.g. more aggressive follow-up) for those 
  groups. This would need careful review before any real-world use.

### Dependencies
- This work depends on the HealthConnect Appointment Dataset and Data 
  Dictionary remaining unchanged in structure for consistency across 
  project weeks.
- Later modelling stages (Week 5 onward) will depend on the target 
  variable and feature decisions made in this document remaining the 
  agreed foundation, unless new evidence justifies revisiting them.
- Coordination with the Data Analytics track may be useful, since 
  their initial analysis and proposed KPIs may surface additional 
  relevant variables or business questions relevant to modelling.